### Implementing baseline models with evasion datasets

In [15]:
import pandas as pd
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

# Define base path to enron2 folder
base_path = "dataset/enron2"

# Load original dataset splits
original_train = pd.read_csv(os.path.join(base_path, "enron2_train.csv"))
original_val = pd.read_csv(os.path.join(base_path, "enron2_val.csv"))
original_test = pd.read_csv(os.path.join(base_path, "enron2_test.csv"))

# Load charswap dataset splits
charswap_train = pd.read_csv(os.path.join(base_path, "charswap/enron2_train_with_charswap.csv"))
charswap_val = pd.read_csv(os.path.join(base_path, "charswap/enron2_val_with_charswap.csv"))
charswap_test = pd.read_csv(os.path.join(base_path, "charswap/enron2_test_with_charswap.csv"))

# Load homoglyph dataset splits
homoglyph_train = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_train_homoglyph.csv"))
homoglyph_val = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_val_homoglyph.csv"))
homoglyph_test = pd.read_csv(os.path.join(base_path, "homoglyph/enron2_test_homoglyph.csv"))

# Load spacing dataset splits
spacing_train = pd.read_csv(os.path.join(base_path, "spacing/enron2_train_spacing.csv"))
spacing_val = pd.read_csv(os.path.join(base_path, "spacing/enron2_val_spacing.csv"))
spacing_test = pd.read_csv(os.path.join(base_path, "spacing/enron2_test_spacing.csv"))

# Check the shape of one dataset to confirm loading
print("Original Train Shape:", original_train.shape)
print(original_train.head())
print("Charswap Train Shape:", charswap_train.shape)
print(charswap_train.head())
print("Homoglyph Train Shape:", homoglyph_train.shape)
print(homoglyph_train.head())
print("Spacing Train Shape:", spacing_train.shape)
print(spacing_train.head())

Original Train Shape: (3727, 2)
                                               email target
0  Subject: membership in the nsf vince : karen m...    ham
1  Subject: holiday gift thank you so much for yo...    ham
2  Subject: re : real options vince , if you take...    ham
3  Subject: men charset = windows - 1252 " > vigo...   spam
4  Subject: vaal medz how t pestilent o save on y...   spam
Charswap Train Shape: (3727, 4)
                                               email target  \
0  Subject: membership in the nsf vince : karen m...    ham   
1  Subject: holiday gift thank you so much for yo...    ham   
2  Subject: re : real options vince , if you take...    ham   
3  Subject: men charset = windows - 1252 " > vigo...   spam   
4  Subject: vaal medz how t pestilent o save on y...   spam   

                                   email_charswapped  was_augmented  
0  Subject: membership in the nsf vince : karen m...          False  
1  Subject: holiday gift thank you so much for yo...     

In [4]:
# Function to rename columns and encode labels
def standardize_dataset(df, modified_text_column=None):
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()
    
    # If there's a modified text column, rename it to 'email_modified'
    if modified_text_column:
        df = df.rename(columns={modified_text_column: "email_modified"})
        # Keep only 'email_modified' and 'target' columns, drop others
        df = df[["email_modified", "target"]]
    else:
        # For original dataset, rename 'email' to 'email_modified'
        df = df.rename(columns={"email": "email_modified"})
    
    # Encode the 'target' column: "ham" -> 0, "spam" -> 1
    le = LabelEncoder()
    df["target"] = le.fit_transform(df["target"])
    
    return df

In [8]:
# Standardize all datasets
# Original dataset (no modified text column)
original_train = standardize_dataset(original_train)
original_val = standardize_dataset(original_val)
original_test = standardize_dataset(original_test)

# Charswap dataset
charswap_train = standardize_dataset(charswap_train, "email_charswapped")
charswap_val = standardize_dataset(charswap_val, "email_charswapped")
charswap_test = standardize_dataset(charswap_test, "email_charswapped")

# Homoglyph dataset
homoglyph_train = standardize_dataset(homoglyph_train, "email_homoglyph")
homoglyph_val = standardize_dataset(homoglyph_val, "email_homoglyph")
homoglyph_test = standardize_dataset(homoglyph_test, "email_homoglyph")

# Spacing dataset
spacing_train = standardize_dataset(spacing_train, "email_spaced")
spacing_val = standardize_dataset(spacing_val, "email_spaced")
spacing_test = standardize_dataset(spacing_test, "email_spaced")

In [9]:
# Check the standardized datasets
print("Standardized Original Train Shape:", original_train.shape)
print(original_train.head())
print("Standardized Charswap Train Shape:", charswap_train.shape)
print(charswap_train.head())
print("Standardized Homoglyph Train Shape:", homoglyph_train.shape)
print(homoglyph_train.head())
print("Standardized Spacing Train Shape:", spacing_train.shape)
print(spacing_train.head())

Standardized Original Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject: holiday gift thank you so much for yo...       0
2  Subject: re : real options vince , if you take...       0
3  Subject: men charset = windows - 1252 " > vigo...       1
4  Subject: vaal medz how t pestilent o save on y...       1
Standardized Charswap Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject: holiday gift thank you so much for yo...       0
2  Subject: re : real options vince , if you take...       0
3  ['Subject: men charset = windows - 1252 "> vig...       1
4  ['Subject: aval medz how t pestilent o svae on...       1
Standardized Homoglyph Train Shape: (3727, 2)
                                      email_modified  target
0  Subject: membership in the nsf vince : karen m...       0
1  Subject

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Function to preprocess and vectorize text data
def preprocess_and_vectorize(train_data, val_data, test_data, text_column="email_modified"):
    vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
    
    # Fit the vectorizer on the training data and transform all splits
    X_train = vectorizer.fit_transform(train_data[text_column])
    X_val = vectorizer.transform(val_data[text_column])
    X_test = vectorizer.transform(test_data[text_column])
    
    # Extract labels
    y_train = train_data["target"]
    y_val = val_data["target"]
    y_test = test_data["target"]
    
    return X_train, X_val, X_test, y_train, y_val, y_test, vectorizer

In [13]:
# Apply preprocessing to each dataset
datasets = {
    "original": (original_train, original_val, original_test),
    "charswap": (charswap_train, charswap_val, charswap_test),
    "homoglyph": (homoglyph_train, homoglyph_val, homoglyph_test),
    "spacing": (spacing_train, spacing_val, spacing_test)
}

# Dictionary to store vectorized data
vectorized_data = {}
for name, (train, val, test) in datasets.items():
    X_train, X_val, X_test, y_train, y_val, y_test, vectorizer = preprocess_and_vectorize(train, val, test)
    vectorized_data[name] = (X_train, X_val, X_test, y_train, y_val, y_test)
    print(f"{name} - X_train shape: {X_train.shape}, X_val shape: {X_val.shape}, X_test shape: {X_test.shape}")

original - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
charswap - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
homoglyph - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)
spacing - X_train shape: (3727, 5000), X_val shape: (932, 5000), X_test shape: (1165, 5000)


In [16]:
# Function to train and evaluate logistic regression
def train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, dataset_name):
    # Initialize and train the model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    # Evaluate on validation set
    y_val_pred = model.predict(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_precision = precision_score(y_val, y_val_pred)
    val_recall = recall_score(y_val, y_val_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    
    # Evaluate on test set
    y_test_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    
    # Print results
    print(f"\nResults for {dataset_name}:")
    print("Validation Set:")
    print(f"Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    print("Test Set:")
    print(f"Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1: {test_f1:.4f}")
    
    return model

In [17]:
# Train and evaluate on each dataset
models = {}
for name, (X_train, X_val, X_test, y_train, y_val, y_test) in vectorized_data.items():
    model = train_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, name)
    models[name] = model


Results for original:
Validation Set:
Accuracy: 0.9871, Precision: 0.9956, Recall: 0.9540, F1: 0.9744
Test Set:
Accuracy: 0.9845, Precision: 0.9930, Recall: 0.9465, F1: 0.9692

Results for charswap:
Validation Set:
Accuracy: 0.9903, Precision: 0.9957, Recall: 0.9665, F1: 0.9809
Test Set:
Accuracy: 0.9914, Precision: 1.0000, Recall: 0.9666, F1: 0.9830

Results for homoglyph:
Validation Set:
Accuracy: 0.9979, Precision: 1.0000, Recall: 0.9916, F1: 0.9958
Test Set:
Accuracy: 0.9940, Precision: 1.0000, Recall: 0.9766, F1: 0.9882

Results for spacing:
Validation Set:
Accuracy: 0.9946, Precision: 1.0000, Recall: 0.9791, F1: 0.9894
Test Set:
Accuracy: 0.9931, Precision: 1.0000, Recall: 0.9732, F1: 0.9864


In [18]:
from sklearn.model_selection import GridSearchCV
# Fine-tuning
def fine_tune_model(X_train, X_val, y_train, y_val, dataset_name):
    param_grid = {"C": [0.01, 0.1, 1, 10, 100]}
    model = LogisticRegression(max_iter=1000)
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring="f1", n_jobs=-1)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    print(f"\nBest parameters for {dataset_name}: {grid_search.best_params_}")
    y_val_pred = best_model.predict(X_val)
    val_f1 = f1_score(y_val, y_val_pred)
    print(f"Best F1-score on validation set for {dataset_name}: {val_f1:.4f}")
    return best_model

In [19]:
# Fine-tune for each dataset
tuned_models = {}
for name, (X_train, X_val, X_test, y_train, y_val, y_test) in vectorized_data.items():
    tuned_model = fine_tune_model(X_train, X_val, y_train, y_val, name)
    tuned_models[name] = tuned_model
    y_test_pred = tuned_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    print(f"Tuned Test Set Results for {name}:")
    print(f"Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1: {test_f1:.4f}")


Best parameters for original: {'C': 100}
Best F1-score on validation set for original: 0.9895
Tuned Test Set Results for original:
Accuracy: 0.9948, Precision: 0.9900, Recall: 0.9900, F1: 0.9900

Best parameters for charswap: {'C': 100}
Best F1-score on validation set for charswap: 0.9896
Tuned Test Set Results for charswap:
Accuracy: 0.9948, Precision: 0.9900, Recall: 0.9900, F1: 0.9900

Best parameters for homoglyph: {'C': 100}
Best F1-score on validation set for homoglyph: 0.9958
Tuned Test Set Results for homoglyph:
Accuracy: 0.9974, Precision: 0.9966, Recall: 0.9933, F1: 0.9950

Best parameters for spacing: {'C': 100}
Best F1-score on validation set for spacing: 0.9958
Tuned Test Set Results for spacing:
Accuracy: 0.9966, Precision: 1.0000, Recall: 0.9866, F1: 0.9933
